# Bigdata processing part I
## 1. Data source, import and export
### 1.1 csv

In [29]:
from pyspark.sql import SparkSession

In [30]:
spark = SparkSession.builder \
    .appName("SparkByExamples")\
    .getOrCreate()

In [ ]:
for conf in spark.sparkContext.getConf().getAll() : 
    print(conf)

In [ ]:
executor_memory = spark.sparkContext.getConf().get("spark.executor.memory")
print(f"Executor Memory: {executor_memory}")

In [ ]:
driver_memory = spark.sparkContext.getConf().get("spark.driver.memory")
print(f"driver Memory: {driver_memory}")

In [ ]:
spark = (SparkSession.builder
        .appName("SparkExample")
        .config("spark.executor.memory", "4g") # 2g default
        .config("spark.driver.memory", "4g") # 2g default
        # .config("spark.executor.instances", 10)
        .getOrCreate())

In [31]:
# BUCKET = "fgao-ensae"
BUCKET = "ematzner-ensae"
FILE_KEY_S3 = "flight-data"
s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}"
s3_path

's3a://ematzner-ensae/flight-data'

In [ ]:
# BUCKET = "fgao-ensae"
BUCKET = "1020cepe01"
FILE_KEY_S3 = "flight-data"
s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}"
s3_path

In [27]:
# optionnel
import os
import s3fs
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })
fs.ls('ematzner-ensae')

['ematzner-ensae/Mllib',
 'ematzner-ensae/Prix20181114.csv',
 'ematzner-ensae/Recommendation',
 'ematzner-ensae/Streaming',
 'ematzner-ensae/dataini.parquet',
 'ematzner-ensae/dataset.parquet',
 'ematzner-ensae/export',
 'ematzner-ensae/flight-data',
 'ematzner-ensae/readmission_avc.parquet',
 'ematzner-ensae/retail-org',
 'ematzner-ensae/top_clients_by_category']

In [33]:
df = (spark.read.format('csv')
      .option("header", True)
      # .load(s3_path+'csv/2014-summary.csv')
      .load(s3_path+'/csv'))
df.count()

747

In [ ]:
df.printSchema()

In [ ]:
df = (spark.read.format('csv')
      .option("header", True)
      .option("inferSchema", True)
      .load(s3_path))

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.types import StructField, StructType, LongType, IntegerType, StringType # we can use IntergerType but better to show the change

**Pourquoi il faut importer les types en spark**

Spark, ce n’est pas juste Python : c’est un système distribué, écrit en Java et Scala. Quand on écrit du code Spark en Python (PySpark), on envoie en réalité des instructions à un moteur Java tournant en arrière-plan (via Py4J). Donc, on ne travaille pas avec des types Python comme int ou str, mais avec des types que Spark connaît et comprend de façon distribuée.

Ces types (StringType, IntegerType, etc.) font partie d’un module spécifique de PySpark : pyspark.sql.types. Python ne connaît pas ces classes de base, donc il faut les importer explicitement comme n’importe quelle autre classe venant d’un module externe.

In [ ]:
my_Schema = StructType(
    [
        StructField("DEST_COUNTRY_NAME", StringType()),
        StructField("ORIGIN_COUNTRY_NAME", StringType()),
        StructField("count", LongType())
    ]
)

In [ ]:
current_shemas = df.schema

In [ ]:
df = (spark.read.format('csv')
      .option("header", True)
      .schema(current_shemas)
      .load(s3_path+'csv/4'))
df.printSchema()
df.count()

#### Parquet

Pas besoin de définir le schéma manuellement car le format Parquet inclut déjà les métadonnées qui décrivent le schéma du fichier. Spark utilise ces métadonnées pour détecter automatiquement les colonnes et leurs types.


Le fichier _SUCCESS dans un répertoire contenant des fichiers Parquet (ou tout autre format de sortie Spark) est un fichier indicateur créé par Apache Spark pour signifier que l'opération d'écriture (ou de sauvegarde) s'est terminée avec succès.

In [ ]:
df_parquet = spark.read.parquet(s3_path+'parquet')

In [ ]:
df_parquet.count()

In [ ]:
df_parquet.show()

In [ ]:
# df_json = spark.read.json(s3_path+'json')
# df_json.show()
# df_json.printSchema()
# df_json.count()

## 2. Spark DF transformations

**[Question 1 : ] Le langage typé fort ou faible** 

REF : https://www.zdnet.fr/lexique-it/le-langage-type-fort-ou-faible-interprete-ou-compile-une-definition-39926151.htm

Les langages fortement typés (comme Java, C++, Rust, etc.) et les langages faiblement typés comme Python ont chacun leurs avantages et inconvénients. Voici une comparaison des deux paradigmes :

1. Avantages des langages fortement typés
a) Détection d'erreurs à la compilation
Avantage : Dans les langages fortement typés, les types de données sont strictement définis, ce qui permet au compilateur de détecter de nombreuses erreurs potentielles avant même l'exécution du programme. Cela réduit les risques de bugs liés à des erreurs de type (par exemple, additionner une chaîne de caractères et un nombre).
Impact : La vérification des types garantit une plus grande robustesse du code, en particulier dans les grandes bases de code.
b) Meilleure optimisation des performances
Avantage : Le compilateur peut effectuer des optimisations basées sur les types de données connus à l'avance. Cela conduit souvent à un code plus performant car le compilateur sait exactement quel type de données sera utilisé et peut optimiser les opérations en conséquence.
Impact : Les langages fortement typés, comme C++ ou Rust, sont souvent préférés pour les applications exigeantes en termes de performance, comme les jeux ou les systèmes embarqués.
Sécurité accrue
c)Avantage : La forte typisation offre plus de sécurité, car elle réduit le risque de certaines erreurs courantes, comme l'utilisation incorrecte de types de données. Par exemple, passer une chaîne de caractères à une fonction qui attend un nombre déclenchera une erreur de compilation.
Impact : Moins d'erreurs runtime (pendant l'exécution), et une meilleure gestion des types rend le code plus sûr.
2. Inconvénients des langages fortement typés
a) Moins de flexibilité
Inconvénient : Dans les langages fortement typés, il est nécessaire de déclarer explicitement les types de toutes les variables, ce qui peut rendre le code plus verbeux et moins flexible. Par exemple, si vous devez changer le type d'une variable, vous devrez modifier toutes les références à cette variable dans le code.
Impact : Cela peut rendre le développement plus lent et plus rigide, surtout pour des applications où les types de données changent souvent.
b) Complexité accrue
Inconvénient : Dans certains langages fortement typés, comme C++, la gestion des types peut devenir complexe, notamment avec des concepts avancés comme la conversion de types, les modèles, ou la surcharge d'opérateurs. Cela peut augmenter la courbe d'apprentissage pour les développeurs débutants.
Impact : Cela peut rendre les langages fortement typés plus difficiles à utiliser dans certains cas, en particulier pour des projets simples ou des prototypes rapides.
c) Temps de développement plus long
3. Avantages de Python (faiblement typé et dynamiquement typé)
a) Flexibilité
b) Productivité accrue
c) Code concis et lisible
4. Inconvénients de Python (faiblement typé)
a) Erreurs de type à l'exécution
En raison du typage dynamique, les erreurs liées aux types ne sont détectées qu'à l'exécution, ce qui peut entraîner des bugs inattendus à des moments critiques. Cela augmente le risque d'erreurs qui n'apparaissent que tardivement dans le cycle de développement. Un débogage plus long et des risques de bugs en production, car certaines erreurs ne sont pas détectées avant l'exécution du programme.
b) Performance plus faible
Inconvénient : L'absence de typage statique signifie que Python doit effectuer des vérifications supplémentaires lors de l'exécution pour déterminer le type de chaque variable. Cela peut rendre les programmes Python plus lents comparés aux langages fortement typés et compilés.

**[Question 2 : ] L'objet Row existe dans Spark mais pas dans Python pur pour plusieurs raisons liées aux besoins spécifiques du framework Apache Spark et à la nature distribuée de ses opérations, contrairement à Python classique.** 

Row objet permet une représentation claire et cohérente des enregistrements sur tous les nœuds du cluster.

In [ ]:
# from pyspark.sql.types import Row
# myRow = Row("Hello", None, 1, True)

# new_row = Row(DEST_COUNTRY_NAME="United States", ORIGIN_COUNTRY_NAME="Hawaii", count=20)
# new_df = spark.createDataFrame([new_row])
# df.union(new_df).count()

### 2.1 Col()

col() faire référence à une colonne spécifique dans un DataFrame

In [ ]:
df.select("DEST_COUNTRY_NAME")
df.selectExpr("DEST_COUNTRY_NAME as destination")
df.select("DEST_COUNTRY_NAME".alias("destination"))

In [ ]:
from pyspark.sql.functions import col

In [ ]:
df.select(col('DEST_COUNTRY_NAME').alias('destination')).show(2)

<!-- ### 2.3 Expressions : select(expr()) and selectExpr -->

In [ ]:
# from pyspark.sql.functions import expr

# df.select(expr("DEST_COUNTRY_NAME as Destination"), "ORIGIN_COUNTRY_NAME").show(2)

# df.selectExpr("DEST_COUNTRY_NAME as Destination", "ORIGIN_COUNTRY_NAME")

### 2.2 Add literals (constant) columns

In [ ]:
from pyspark.sql.functions import lit

In [ ]:
df.select(expr("*"), lit(10).alias('REF')).show(2)

In [ ]:
df.withColumn("REF", lit(50)).show(2)

### 2.3 Basic manipulations

In [ ]:
df.withColumn("count", col('count').cast("float")).show(2)

In [ ]:
df.select("DEST_COUNTRY_NAME").distinct().count()

In [ ]:
from pyspark.sql.functions import desc
df.sort("count", desc("ORIGIN_COUNTRY_NAME")).show(10)

In [ ]:
df.where('count >= 15')
df.filter('count >= 15')

In [ ]:
df.where(col('count') >= 15)
df.filter(col('count') >= 15)

In [ ]:
# .when().otherwise()
from pyspark.sql.functions import when
df.withColumn("other_count", 
              when(col("count") < 30, col("count") + 1000).otherwise(col("count") - 500)).show(2)

### 3.4 Aggregation
* df.groupBy().unique_agg_function()
* df.groupBy().agg(func_1, func_2)

In [ ]:
from pyspark.sql.functions import avg, sum, max, min, count, countDistinct, collect_list, collect_set

In [ ]:
df.groupby("ORIGIN_COUNTRY_NAME").sum("count").show(5)

In [ ]:
df.groupby("ORIGIN_COUNTRY_NAME").avg("count").show(5)

In [ ]:
df.groupby("ORIGIN_COUNTRY_NAME").agg(sum("count"), avg("count"), max("count")).show(5)

In [ ]:
df.groupby('DEST_COUNTRY_NAME').agg(collect_list('ORIGIN_COUNTRY_NAME'), collect_set('ORIGIN_COUNTRY_NAME')).show(10, False)

### 3.5 Random Sample and split

In [ ]:
df.sample(withReplacement= False, fraction = 0.1, seed = 42).count()

In [ ]:
train, test = df.randomSplit([.75, .25], seed = 42)
print(train.count())
print(test.count())

In [ ]:
train.union(test).count()

### 3.6 .repartition() & .coalesce()

When Spark uses a parallelism level of 200 partitions, but your cluster doesn’t have 200 nodes or even close to that, the following happens:

* If your cluster has fewer nodes than partitions, Spark will execute the partitions on the available nodes. Each node processes multiple partitions. For example, if you have 10 nodes, and you have 200 partitions, each node may process 20 partitions in parallel.
* If you have a cluster with 10 nodes and each node has 8 cores, you have a total of 80 cores. If you set the number of partitions to 200:
> 1. Spark will schedule up to 80 tasks concurrently (one per core).
> 2. The remaining tasks will be queued until the running tasks complete and resources become available.

In [ ]:
print(df.rdd.getNumPartitions())

df = df.repartition(5)
df.rdd.getNumPartitions()

In [ ]:
from pyspark.sql.functions import spark_partition_id, count


In [ ]:
df.withColumn('partitionID', spark_partition_id()).show(200)

In [ ]:
df.withColumn('partitionID', spark_partition_id()).groupby('partitionID').count().show()

In [ ]:
df = df.repartition(5, 'DEST_COUNTRY_NAME')

In [ ]:
df.withColumn('partitionID', spark_partition_id()).groupby('partitionID').count().show()

In [ ]:
df= df.coalesce(2)

In [ ]:
df.rdd.getNumPartitions()

df.withColumn('partitionID', spark_partition_id()).groupby('partitionID').count().show()

# It does not provide specific control over which partitions are merged, and the merging logic is handled internally by Spark to minimize data movement and optimize performance.

In [ ]:
df_repartitioned = df.repartition(10, 'col1', 'col2', 'col3')

Pourquoi utiliser repartition() ?
* Optimisation des jointures : Si vous avez besoin de faire des jointures sur les colonnes col1, col2, et col3, avoir un DataFrame pré-repartitionné peut réduire le mouvement de données entre les partitions.
* GroupBy plus efficace : Pour des opérations comme groupBy, repartitionner sur les colonnes clés peut accélérer le processus en minimisant le "shuffle" des données.

### 3.7 windows function (optionnel)

In [ ]:
data = [
    ("Alice", "HR", 500, "Alice is a hardworking employee." ),
    ("Bob", "HR", 1000, "Bob has a strong performance in HR."),
    ("Charlie", "IT", 1500, "Charlie is a valuable IT team member."),
    ("David", "IT", 2000, "David consistently excels in IT."),
    ("David", "IT", 2000, "This is another David"),
    ("Eve", "IT", 3000, "Eve is a top performer in the IT department.")
]

# Create DataFrame
df = spark.createDataFrame(data, ["employee", "department", "sales", "description"])

In [ ]:
from pyspark.sql.window import Window

In [ ]:
windowSpec = Window.partitionBy("department").orderBy("sales")

In [ ]:
type(windowSpec)

In [ ]:
from pyspark.sql.functions import row_number

In [ ]:
df.withColumn('row', row_number().over(windowSpec)).show()

# syntaxe : function().over(windowSpec)

In [ ]:
from pyspark.sql.functions import rank, dense_rank, cume_dist, lag, lead, collect_list, collect_set

# lag(column, offset=1, default=None)

In [ ]:
df.withColumn('rank', rank().over(windowSpec))\
.withColumn('dense_rank', dense_rank().over(windowSpec))\
.withColumn('cume_dist', cume_dist().over(windowSpec))\
.withColumn('lag', lag("sales", 1).over(windowSpec))\
.withColumn('lead', lead("sales", 1).over(windowSpec))\
.withColumn('list_of_employees', collect_list('employee').over(windowSpec)).show(10, False)

# sans utiliser windows function, il faut créer une aggrégation et faire de la jointure

### 4. User-defined functions (UDF)

In [ ]:
data = [
    ("Alice", "HR", 500, "Alice is a hardworking employee." ),
    ("Bob", "HR", 1000, "Bob has a strong performance in HR."),
    ("Charlie", "IT", 1500, "Charlie is a valuable IT team member."),
    ("David", "IT", 2000, "David consistently excels in IT."),
    ("David", "IT", 2000, "This is another David"),
    ("Eve", "IT", 3000, "Eve is a top performer in the IT department.")
]

# Create DataFrame
data = spark.createDataFrame(data, ["employee", "department", "sales", "description"])

In [ ]:
def counting_tokens(text):
    words = text.split()
    token_count = {word: words.count(word) for word in set(words)}
    return token_count

In [ ]:
sentence = 'Hello I am learning learning python'
counting_tokens(sentence)

Now that we’ve created these functions and tested them, we need to register them with Spark so that we can use them on all of our worker machines. Spark will serialize the function on the driver and transfer it over the network to all executor processes. This happens regardless of language. When you use the function, there are essentially two different things that occur. If the function is written in Scala or Java, you can use it within the Java Virtual Machine (JVM). This means that there will be little performance penalty aside from the fact that you can’t take advantage of code generation capabilities that Spark has for built-in functions.

If the function is written in Python, something quite different happens. Spark starts a Python process on the worker, serializes all of the data to a format that Python can understand (remember, it was in the JVM earlier), executes the function row by row on that data in the Python process, and then finally returns the results of the row operations to the JVM and Spark.

**why pyspark functions works with performance but not Python UDF function?**

https://stackoverflow.com/questions/38296609/spark-functions-vs-udf-performance

Spark DataFrame is natively a JVM structure and standard access methods are implemented by simple calls to Java API. UDF from the other hand are implemented in Python and require moving data back and forth.

While PySpark in general requires data movements between JVM and Python, in case of low level RDD API it typically doesn't require expensive serde activity. Spark SQL adds additional cost of serialization and serialization as well cost of moving data from and to unsafe representation on JVM. The later one is specific to all UDFs (Python, Scala and Java) but the former one is specific to non-native languages.

Unlike UDFs, **Spark SQL functions** such as pyspark function operate directly on JVM and typically are well integrated with both Catalyst and Tungsten. It means these can be optimized in the execution plan and most of the time can benefit from codgen and other Tungsten optimizations. Moreover these can operate on data in its "native" representation.

In [ ]:
# This don't work
# df.withColumn('count', counting_tokens('description')).show()

from pyspark.sql.functions import udf
from pyspark.sql.types import MapType, IntegerType, StringType
counting_tokens_udf = udf(counting_tokens, MapType(StringType(), IntegerType()))

In [ ]:
from pyspark.sql.functions import col, udf

In [ ]:
df.withColumn('count', counting_tokens_udf('description')).show(10, False)

En dessus c'est un autre exemple. A peaufiner plus tard

In [ ]:
# Step 1: Define the window spec to calculate the running total of sales per department
# windowSpec = Window.partitionBy("department").orderBy("sales").rowsBetween(Window.unboundedPreceding, Window.currentRow)

In [ ]:
windowSpec2 = Window.partitionBy("department").orderBy("sales").rowsBetween(Window.unboundedPreceding, Window.currentRow)

In [ ]:
df_with_running_total = df.withColumn("running_total", sum("sales").over(windowSpec))
df_with_running_total.show()

In [ ]:
# Step 2: Calculate the running total of sales for each department
df_with_running_total = df.withColumn("running_total", sum("sales").over(windowSpec2))
df_with_running_total.show()

In [ ]:
# Step 3: Define the custom discount UDF
def apply_bonus(total):
    if total < 2000:
        return total * 0.05  # 5% discount
    elif 2000 <= total <= 4000:
        return total * 0.10  # 10% discount
    else:
        return total * 0.15  # 15% discount

In [ ]:
df_with_running_total.show()

In [ ]:
# Register UDF
bonus_udf = udf(apply_bonus, DoubleType()) # not obligatoire, mais 

# Step 4: Apply the discount based on the running total
df_with_discount = df_with_running_total.withColumn("department_bonus", bonus_udf(col("running_total")))

# Show the result
df_with_discount.show(truncate=False)

Type Safety: Defining DoubleType() ensures that PySpark handles the output as a double-precision floating-point number, which helps avoid type-related errors or inconsistencies.
Schema Definition: When you define the return type, PySpark knows the exact type of data being processed. This can be crucial for downstream operations that depend on specific data types, such as aggregations or joins.
Performance: PySpark can optimize operations better when it knows the data type in advance. While the difference might be minor, specifying types helps PySpark make efficient execution plans.
Readability: Explicitly defining the return type makes the code clearer and more maintainable. It documents the expected output type for anyone reading the code.

If you omit DoubleType(), PySpark may infer the type based on the function's return values. While this often works, it’s not always reliable, especially if the UDF can return multiple types or if there are type conversions happening in your code.

## 5. Join
### 5.1 Join type

In [ ]:
person = spark.createDataFrame([
(0, "Bill Chambers", 0, [100]),
(1, "Matei Zaharia", 1, [500, 250, 100]),
(2, "Michael Armbrust", 1, [250, 100])])\
.toDF("id", "name", "graduate_program", "spark_status")
graduateProgram = spark.createDataFrame([
(0, "Masters", "School of Information", "UC Berkeley"),
(2, "Masters", "EECS", "UC Berkeley"),
(1, "Ph.D.", "EECS", "UC Berkeley")])\
.toDF("id", "degree", "department", "school")
sparkStatus = spark.createDataFrame([
(500, "Vice President"),
(250, "PMC Member"),
(100, "Contributor")])\
.toDF("id", "status")

person.show()
graduateProgram.show(5, False)
sparkStatus.show()

### 5.2 Force a broadcast join

In [ ]:
person.rdd.getNumPartitions() # 2
graduateProgram.rdd.getNumPartitions() # 2

graduateProgram = graduateProgram.repartition(3)

person.rdd.getNumPartitions()

In [ ]:
from pyspark.sql.functions import broadcast

In [ ]:
person.join(broadcast(graduateProgram), person.graduate_program == graduateProgram.id, how = 'inner').rdd.getNumPartitions()
# 'inner', 'outer', 'full', 'fullouter', 'full_outer', 'leftouter', 'left', 'left_outer', 'rightouter', 'right', 'right_outer', 'leftsemi' = 'left_semi' = 'semi', 'leftanti', 'left_anti', 'anti', 'cross'.

EXO : Montrer l'inverse

In [ ]:
graduateProgram.join(broadcast(person), person.graduate_program == graduateProgram.id, how = 'inner').rdd.getNumPartitions()

## Exo de manipulation de donnée

In [14]:
FILE_KEY_S3 = "Recommendation/sample_movielens_ratings.txt"
s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}"
s3_path

's3a://ematzner-ensae/Recommendation/sample_movielens_ratings.txt'

In [6]:
spark.read.text(s3_path).show()

+--------------------+
|               value|
+--------------------+
| 0::2::3::1424380312|
| 0::3::1::1424380312|
| 0::5::2::1424380312|
| 0::9::4::1424380312|
|0::11::1::1424380312|
|0::12::2::1424380312|
|0::15::1::1424380312|
|0::17::1::1424380312|
|0::19::1::1424380312|
|0::21::1::1424380312|
|0::23::1::1424380312|
|0::26::3::1424380312|
|0::27::1::1424380312|
|0::28::1::1424380312|
|0::29::1::1424380312|
|0::30::1::1424380312|
|0::31::1::1424380312|
|0::34::1::1424380312|
|0::37::1::1424380312|
|0::41::2::1424380312|
+--------------------+
only showing top 20 rows



In [16]:
from pyspark.sql.functions import split, from_unixtime, col, to_timestamp

In [17]:
ratings = spark.read.text(s3_path).select(split("value", "::").alias('columns')).select(
    col("columns")[0].alias("userID").cast('int'),
    col("columns")[1].alias("movieID").cast('int'),
    col("columns")[2].alias("rating").cast('float'),
    col("columns")[3].alias("ts")
).withColumn('date_time', to_timestamp(from_unixtime("ts"), 'yyyy-MM-dd HH:mm:ss'))

In [18]:
ratings.printSchema()

root
 |-- userID: integer (nullable = true)
 |-- movieID: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- ts: string (nullable = true)
 |-- date_time: timestamp (nullable = true)



In [19]:
ratings.show()

+------+-------+------+----------+-------------------+
|userID|movieID|rating|        ts|          date_time|
+------+-------+------+----------+-------------------+
|     0|      2|   3.0|1424380312|2015-02-19 21:11:52|
|     0|      3|   1.0|1424380312|2015-02-19 21:11:52|
|     0|      5|   2.0|1424380312|2015-02-19 21:11:52|
|     0|      9|   4.0|1424380312|2015-02-19 21:11:52|
|     0|     11|   1.0|1424380312|2015-02-19 21:11:52|
|     0|     12|   2.0|1424380312|2015-02-19 21:11:52|
|     0|     15|   1.0|1424380312|2015-02-19 21:11:52|
|     0|     17|   1.0|1424380312|2015-02-19 21:11:52|
|     0|     19|   1.0|1424380312|2015-02-19 21:11:52|
|     0|     21|   1.0|1424380312|2015-02-19 21:11:52|
|     0|     23|   1.0|1424380312|2015-02-19 21:11:52|
|     0|     26|   3.0|1424380312|2015-02-19 21:11:52|
|     0|     27|   1.0|1424380312|2015-02-19 21:11:52|
|     0|     28|   1.0|1424380312|2015-02-19 21:11:52|
|     0|     29|   1.0|1424380312|2015-02-19 21:11:52|
|     0|  

In [20]:
FILE_KEY_S3 = "Recommendation/ratings.parquet"
s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}"
s3_path

's3a://ematzner-ensae/Recommendation/ratings.parquet'

In [21]:
ratings.write.parquet(s3_path)

In [22]:
spark.read.parquet(s3_path)

DataFrame[userID: int, movieID: int, rating: float, ts: string, date_time: timestamp]

In [28]:
spark.stop()